In [6]:
import sys
print(sys.executable)

!{sys.executable} -m pip install pandas numpy pyarrow scikit-learn torch

c:\Users\uvvss\miniconda3\python.exe
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 9.7 MB/s eta 0:00:01
   ---------------- ----------------------- 3.9/9.7 MB 9.5 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.7 MB 9.6 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.7 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 

In [1]:
import pandas as pd
import numpy as np

print("Pandas OK:", pd.__version__)

Pandas OK: 3.0.2


In [6]:
%pip install pandas numpy pyarrow scikit-learn scipy torch matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

import random
import math
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.optimize import Bounds, LinearConstraint, minimize
from scipy.spatial import ConvexHull, QhullError

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

OSError: [WinError 127] The specified procedure could not be found. Error loading "c:\Users\uvvss\miniconda3\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [4]:
class ROEnv:
    def __init__(self, df):
        self.df = df.copy().sort_values("Timestamp").reset_index(drop=True)
        self.episode_len = EPISODE_LEN

    def reset(self):
        self.start = np.random.randint(K_LAGS, len(self.df) - self.episode_len - 1)
        self.t = self.start
        self.steps = 0

        self.history = self.df.iloc[self.t-K_LAGS:self.t].copy().reset_index(drop=True)
        self.prev_qp = float(self.history.iloc[-1][CONTROL_COL])

        episode_data = self.df.iloc[self.start:self.start+self.episode_len]
        self.target_volume = float(np.sum(episode_data[CONTROL_COL] * DT_HOURS))
        self.w_cum = 0.0

        return self._get_state()

    def _get_state(self):
        row = self.history.iloc[-1]

        qp = row[CONTROL_COL]
        qc = row[QC_COL]
        qr = row[QR_COL]
        power = surrogate_predict_power(self.history)
        tariff = self.df.iloc[self.t]["tariff"]

        water_progress = self.w_cum / self.target_volume
        time_remaining = (self.episode_len - self.steps) / self.episode_len

        state = np.array([
            qp,
            qc,
            qr,
            power,
            tariff,
            water_progress,
            time_remaining,
            self.prev_qp
        ], dtype=np.float32)

        return state

    def step(self, raw_action):
        # Step 2: RL proposes raw action
        raw_action = float(np.clip(raw_action, -1, 1))

        # convert [-1, 1] to [1, 4] gpm
        qp_raw = QP_MIN + (raw_action + 1) * 0.5 * (QP_MAX - QP_MIN)

        # Step 3: safety layer refines action
        qp_star = refine_action_qp(qp_raw, self.prev_qp)

        # Step 4: apply first optimized action
        next_row = self.df.iloc[self.t].copy()
        next_row[CONTROL_COL] = qp_star

        self.history = pd.concat(
            [self.history, pd.DataFrame([next_row])],
            ignore_index=True
        ).iloc[-K_LAGS:].reset_index(drop=True)

        # Step 5: system response from surrogate model
        pred_power = surrogate_predict_power(self.history)

        water = qp_star * DT_HOURS
        self.w_cum += water

        tariff = next_row["tariff"]
        cost = tariff * pred_power * DT_HOURS

        # backlog penalty
        expected_volume = self.target_volume * ((self.steps + 1) / self.episode_len)
        backlog = max(0, expected_volume - self.w_cum)

        # smoothness penalty
        smooth_penalty = (qp_star - self.prev_qp) ** 2

        # Step 6: reward
        reward = (
            -LAMBDA_COST * cost
            -LAMBDA_BACKLOG * backlog
            -LAMBDA_SMOOTH * smooth_penalty
        )

        self.prev_qp = qp_star
        self.t += 1
        self.steps += 1

        done = self.steps >= self.episode_len

        next_state = self._get_state()

        info = {
            "qp_raw": qp_raw,
            "qp_star": qp_star,
            "power": pred_power,
            "cost": cost,
            "water": water,
            "w_cum": self.w_cum,
            "target_volume": self.target_volume,
            "backlog": backlog,
            "smooth_penalty": smooth_penalty
        }

        return next_state, reward, done, info

In [ ]:
env = ROEnv(df_all)

state = env.reset()
print("Initial state:", state)

next_state, reward, done, info = env.step(0.0)

print("Reward:", reward)
print("Done:", done)
print(info)

NameError: name 'EPISODE_LEN' is not defined